In [4]:
!pip install yfinance

!pip install scikit-learn
#version 0.0.post1
!pip install hmmlearn
#version 0.2.8
!pip install plotly

In [104]:
import numpy as np
# from sklearn.cluster import KMeans
import pandas as pd
from hmmlearn.hmm import GaussianHMM
import plotly.graph_objects as go
from plotly.graph_objs.scatter.marker import Line
from plotly.subplots import make_subplots
import plotly.express as px
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture
import warnings
import math
import yfinance as yf

In [11]:
tickers = yf.Tickers("^GSPC ^VIX ^FTSE ^N225 ^HSI").download(period=None, interval='1d', start='2006-01-01', end='2026-01-01')

tickers
# yf.download(['^HSI', '^GSPC', '^FTSE','^N225', '^VIX' ], period='1mo')

[*********************100%***********************]  5 of 5 completed


Price             Close                                                  \
Ticker            ^FTSE        ^GSPC          ^HSI         ^N225   ^VIX   
Date                                                                      
2006-01-03  5681.500000  1268.800049  14944.769531           NaN  11.14   
2006-01-04  5714.600098  1273.459961  15200.059570  16361.540039  11.37   
2006-01-05  5691.200195  1273.479980  15271.129883  16425.369141  11.31   
2006-01-06  5731.799805  1285.449951  15344.440430  16428.210938  11.00   
2006-01-09  5731.500000  1290.150024  15547.429688           NaN  11.13   
...                 ...          ...           ...           ...    ...   
2025-12-25          NaN          NaN           NaN  50407.789062    NaN   
2025-12-26          NaN  6929.939941           NaN  50750.390625  13.60   
2025-12-29  9866.500000  6905.740234  25635.230469  50526.921875  14.20   
2025-12-30  9940.700195  6896.240234  25854.599609  50339.480469  14.33   
2025-12-31  9931.400391  6845.500000  25630.539062           NaN  14.95   

Price      Dividends                        ... Stock Splits                   \
Ticker         ^FTSE ^GSPC ^HSI ^N225 ^VIX  ...        ^FTSE ^GSPC ^HSI ^N225   
Date                                        ...                                 
2006-01-03       0.0   0.0  0.0   NaN  0.0  ...          0.0   0.0  0.0   NaN   
2006-01-04       0.0   0.0  0.0   0.0  0.0  ...          0.0   0.0  0.0   0.0   
2006-01-05       0.0   0.0  0.0   0.0  0.0  ...          0.0   0.0  0.0   0.0   
2006-01-06       0.0   0.0  0.0   0.0  0.0  ...          0.0   0.0  0.0   0.0   
2006-01-09       0.0   0.0  0.0   NaN  0.0  ...          0.0   0.0  0.0   NaN   
...              ...   ...  ...   ...  ...  ...          ...   ...  ...   ...   
2025-12-25       NaN   NaN  NaN   0.0  NaN  ...          NaN   NaN  NaN   0.0   
2025-12-26       NaN   0.0  NaN   0.0  0.0  ...          NaN   0.0  NaN   0.0   
2025-12-29       0.0   0.0  0.0   0.0  0.0  ...          0.0   0.0  0.0   0.0   
2025-12-30       0.0   0.0  0.0   0.0  0.0  ...          0.0   0.0  0.0   0.0   
2025-12-31       0.0   0.0  0.0   NaN  0.0  ...          0.0   0.0  0.0   NaN   

Price                  Volume                                                
Ticker     ^VIX         ^FTSE         ^GSPC          ^HSI        ^N225 ^VIX  
Date                                                                         
2006-01-03  0.0  1.838005e+09  2.554570e+09  2.504000e+08          NaN  0.0  
2006-01-04  0.0  2.005345e+09  2.515330e+09  5.652000e+08   94300000.0  0.0  
2006-01-05  0.0  1.862547e+09  2.433340e+09  5.143000e+08  164500000.0  0.0  
2006-01-06  0.0  1.894952e+09  2.446560e+09  3.977000e+08  170400000.0  0.0  
2006-01-09  0.0  1.690826e+09  2.301490e+09  5.815000e+08          NaN  0.0  
...         ...           ...           ...           ...          ...  ...  
2025-12-25  NaN           NaN           NaN           NaN   58300000.0  NaN  
2025-12-26  0.0           NaN  2.586550e+09           NaN   81100000.0  0.0  
2025-12-29  0.0  3.478967e+08  3.541750e+09  2.931600e+09   98000000.0  0.0  
2025-12-30  0.0  3.647421e+08  3.309930e+09  3.138100e+09   93700000.0  0.0  
2025-12-31  0.0  1.760453e+08  3.261830e+09  1.651000e+09          NaN  0.0  

[5200 rows x 35 columns]

In [ ]:
# Pass the index data into a df

df = pd.DataFrame(tickers['Close'])

spy = pd.DataFrame(df['^FTSE']).rename(columns={'^FTSE':"Close"}).dropna()
vix = pd.DataFrame(df['^VIX']).rename(columns={'^VIX':"Close"}).dropna()
gspc = pd.DataFrame(df['^GSPC']).rename(columns={'^GSPC':"Close"}).dropna()
hsi = pd.DataFrame(df['^HSI']).rename(columns={'^HSI':"Close"}).dropna()
n225 = pd.DataFrame(df['^N225']).rename(columns={'^N225':"Close"}).dropna()
# vix.columns = ["Date", "Close"]
print(vix,spy,gspc,hsi,n225)


            Close
Date             
2006-01-03  11.14
2006-01-04  11.37
2006-01-05  11.31
2006-01-06  11.00
2006-01-09  11.13
...           ...
2025-12-24  13.47
2025-12-26  13.60
2025-12-29  14.20
2025-12-30  14.33
2025-12-31  14.95

[5031 rows x 1 columns]                   Close
Date                   
2006-01-03  5681.500000
2006-01-04  5714.600098
2006-01-05  5691.200195
2006-01-06  5731.799805
2006-01-09  5731.500000
...                 ...
2025-12-23  9889.200195
2025-12-24  9870.700195
2025-12-29  9866.500000
2025-12-30  9940.700195
2025-12-31  9931.400391

[5051 rows x 1 columns]                   Close
Date                   
2006-01-03  1268.800049
2006-01-04  1273.459961
2006-01-05  1273.479980
2006-01-06  1285.449951
2006-01-09  1290.150024
...                 ...
2025-12-24  6932.049805
2025-12-26  6929.939941
2025-12-29  6905.740234
2025-12-30  6896.240234
2025-12-31  6845.500000

[5031 rows x 1 columns]                    Close
Date                    
2006-01-03  14944

In [57]:
# Calculate the log return on the moving average this reduces the noise by normalising the returns in the data

def prepare_data(prices, ma_period):
    
    prices['m_a'] = prices.rolling(ma_period).mean()
    prices['log_return'] = np.log(prices['m_a']/prices['m_a'].shift(1)).dropna()
    prices.dropna(inplace = True)
    
    prices_array = np.array([[p] for p in prices['log_return'].values])
    
    return prices, prices_array


    
    
    

In [84]:
vix_data = prepare_data(vix[["Close"]],7)
spy_data = prepare_data(spy[['Close']], 7)
n225_data = prepare_data(n225[['Close']], 7)
gspc_data = prepare_data(gspc[['Close']], 7)
hsi_data = prepare_data(hsi[["Close"]], 7)

print(vix_data[1])

[[ 0.0007714 ]
 [-0.00180088]
 [ 0.0076953 ]
 ...
 [-0.03325862]
 [-0.02543699]
 [ 0.00040564]]


In [60]:
# Initialze the regime detection using Hidden Markov Model and Gausian Mixture Model

In [62]:
class RegimeDetection:
    
    def initialise_model(self, model, params):
        for parameter, value in params.items():
            setattr(model, parameter, value)
        return model
    
    def hmm_regime(self, input_data, params):
        hmm_model = self.initialise_model(GaussianHMM(), params).fit(input_data)
        return hmm_model
    
    def gmm_regime(self, input_data, params):
        gmm_model = self.initialise_model(GaussianMixture(), params).fit(input_data)
        return gmm_model

In [63]:
# This function plots the hidden states found in the data

def plot_hidden_states(hidden_states, prices_df,symbol):
    colors = ["blue","green"]
    
    # Get the number of uniques states
    n_components = len(np.unique(hidden_states))
    fig = go.Figure()
    
    for i in range(n_components):
        
        # Assign the mask when the hidden state in the state component
        mask = hidden_states == i
        
        print('Number of observations for State ', i,":", len(prices_df.index[mask]))
        
        fig.add_trace(go.Scatter(x=prices_df.index[mask], y=prices_df['Close'][mask],
                    mode='markers', name='Hidden State ' + str(i), marker=dict(size=4,color=colors[i])))
        
    fig.update_layout(title=symbol,height=400, width=900, legend=dict(yanchor="top", y=0.99, xanchor="left",x=0.01),margin=dict(l=20, r=20, t=20, b=20)).show()

In [64]:
regime_detection = RegimeDetection()

In [ ]:
# market regime prediction using the Guassian Mixture model
params = {'n_components':2, 'covariance_type': 'full', 'max_iter': 100000, 'n_init': 30,'init_params': 'kmeans', 'random_state':100}

vix_gmm_model = regime_detection.gmm_regime(vix_data[1],params)
vix_states = vix_gmm_model.predict(vix_data[1])
plot_hidden_states(np.array(vix_states), vix_data[0][['Close']],"VIX")

spy_gmm_model = regime_detection.gmm_regime(spy_data[1],params)
spy_states = spy_gmm_model.predict(spy_data[1])
plot_hidden_states(np.array(spy_states), spy_data[0][['Close']],"SPY")

n225_gmm_model = regime_detection.gmm_regime(n225_data[1],params)
n225_states = n225_gmm_model.predict(n225_data[1])
plot_hidden_states(np.array(n225_states), n225_data[0][['Close']],"N225")

Number of observations for State  0 : 4348
Number of observations for State  1 : 676


Number of observations for State  0 : 4716
Number of observations for State  1 : 328


Number of observations for State  0 : 4401
Number of observations for State  1 : 485


In [93]:
params = {'n_components':2, 'covariance_type':"full", 'random_state':100}

vix_hmm_model = regime_detection.hmm_regime(vix_data[1],params)
vix_hmm_state = vix_hmm_model.predict(vix_data[1])
plot_hidden_states(np.array(vix_hmm_state), vix_data[0][['Close']],"VIX")

spy_hmm_model = regime_detection.hmm_regime(spy_data[1],params)
spy_hmm_states = spy_hmm_model.predict(spy_data[1])
plot_hidden_states(np.array(spy_hmm_states), spy_data[0][['Close']],"SPY")

n225_hmm_model = regime_detection.hmm_regime(n225_data[1],params)
n225_hmm_states = n225_hmm_model.predict(n225_data[1])
plot_hidden_states(np.array(n225_hmm_states), n225_data[0][['Close']],"N225")

Number of observations for State  0 : 1171
Number of observations for State  1 : 3853


Number of observations for State  0 : 467
Number of observations for State  1 : 4577


Number of observations for State  0 : 660
Number of observations for State  1 : 4226


In [89]:
#   Implement feed forwarding to prevent  over fitting and lookback-bias

In [90]:
def feed_forward_training(model, params, prices, split_index, retrain_step):
    print((model, params, prices,split_index, retrain_step))
    
    # Split data into trains and test data
    init_train_data = prices[:split_index]
    test_data = prices[split_index:]
    
    # Train data based on the model
    rd_model = model(init_train_data, params)
    
    # State prediction array
    states_pred = []
    
    for i in range(math.ceil(len(test_data))):
        
        # Move the train data by one step
        split_index += 1
        
        # based on the trained data predict to current index which is i steps from the last training data
        pred = rd_model.predict(prices[:split_index]).tolist()
    
        # Append latest prediction to states prediction array
        states_pred.append(pred[-1])
        
        # Retrain based on the new data
        if i % retrain_step == 0:
            rd_model = model(prices[:split_index], params)
    
    return states_pred 
        
        

In [105]:
spy_hmm_model = regime_detection.hmm_regime
vix_hmm_model = regime_detection.hmm_regime
n225_hmm_model = regime_detection.hmm_regime
params = {'n_components':2, 'covariance_type':"full", 'random_state':100}

# Split index from 2016 to predict state

split_index= np.where(spy_data[0].index > "'2016-01-01'")[0][0]

print(split_index)

2519


In [110]:
spy_states_pred_hmm = feed_forward_training(spy_hmm_model,params,spy_data[1],split_index, 20)
vix_states_pred_hmm = feed_forward_training(vix_hmm_model,params,vix_data[1],split_index, 20)
n225_states_pred_hmm = feed_forward_training(n225_hmm_model,params,n225_data[1],split_index, 20)

print(spy_states_pred_hmm)
plot_hidden_states(np.array(spy_states_pred_hmm), spy_data[0][['Close']][split_index:],"SPY")
plot_hidden_states(np.array(vix_states_pred_hmm), vix_data[0][['Close']][split_index:],"VIX")
plot_hidden_states(np.array(n225_states_pred_hmm), n225_data[0][['Close']][split_index:],"N225")

(<bound method RegimeDetection.hmm_regime of <__main__.RegimeDetection object at 0x000001C74B718C20>>, {'n_components': 2, 'covariance_type': 'full', 'random_state': 100}, array([[ 1.34258060e-03],
       [-8.99511692e-05],
       [ 1.22360830e-03],
       ...,
       [ 2.63673678e-03],
       [ 2.40863037e-03],
       [ 1.35231472e-03]], shape=(5044, 1)), np.int64(2519), 20)
(<bound method RegimeDetection.hmm_regime of <__main__.RegimeDetection object at 0x000001C74B718C20>>, {'n_components': 2, 'covariance_type': 'full', 'random_state': 100}, array([[ 0.0007714 ],
       [-0.00180088],
       [ 0.0076953 ],
       ...,
       [-0.03325862],
       [-0.02543699],
       [ 0.00040564]], shape=(5024, 1)), np.int64(2519), 20)
(<bound method RegimeDetection.hmm_regime of <__main__.RegimeDetection object at 0x000001C74B718C20>>, {'n_components': 2, 'covariance_type': 'full', 'random_state': 100}, array([[-0.00081628],
       [-0.005424  ],
       [-0.00959039],
       ...,
       [ 0.00353

Number of observations for State  0 : 592
Number of observations for State  1 : 1913


Number of observations for State  0 : 179
Number of observations for State  1 : 2188


In [141]:
spy_data_states = pd.DataFrame(spy_data[0][split_index:]['Close'])
spy_data_states['State'] = spy_states_pred_hmm
spy_data_states

vix_data_states = pd.DataFrame(vix_data[0][split_index:]['Close'])
vix_data_states['State'] = vix_states_pred_hmm
vix_data_states

n225_data_states = pd.DataFrame(n225_data[0][split_index:]['Close'])
n225_data_states['State'] = n225_states_pred_hmm
n225_data_states

,Close,State
Date,,
2016-04-25,17439.300781,1
2016-04-26,17353.279297,1
2016-04-27,17290.490234,1
2016-04-28,16666.050781,1
2016-05-02,16147.379883,1
...,...,...
2025-12-24,50344.101562,1
2025-12-25,50407.789062,1
2025-12-26,50750.390625,1


In this section, we implement a simple investment strategy based on the predicted hidden states, which is generating:

A long signal when the market is expected to stay in a normal state.

A short signal when the market is in a crash or high volatility state

In [142]:
# Calculate the daily PnL

vix_data_states['P&L_daily'] = np.log(vix_data_states['Close']/vix_data_states['Close'].shift(1)).dropna()
spy_data_states['P&L_daily'] = np.log(spy_data_states['Close']/spy_data_states['Close'].shift(1)).dropna()
n225_data_states['P&L_daily'] = np.log(n225_data_states['Close']/n225_data_states['Close'].shift(1)).dropna()

print(vix_data_states,spy_data_states,n225_data_states)

                Close  State  P&L_daily
Date                                   
2016-01-15  27.020000      0        NaN
2016-01-19  26.049999      0  -0.036560
2016-01-20  27.590000      0   0.057436
2016-01-21  26.690001      0  -0.033164
2016-01-22  22.340000      1  -0.177910
...               ...    ...        ...
2025-12-24  13.470000      1  -0.038592
2025-12-26  13.600000      1   0.009605
2025-12-29  14.200000      1   0.043172
2025-12-30  14.330000      1   0.009113
2025-12-31  14.950000      1   0.042356

[2505 rows x 3 columns]                   Close  State  P&L_daily
Date                                     
2016-01-04  6093.399902      1        NaN
2016-01-05  6137.200195      1   0.007162
2016-01-06  6073.399902      1  -0.010450
2016-01-07  5954.100098      1  -0.019838
2016-01-08  5912.399902      1  -0.007028
...                 ...    ...        ...
2025-12-23  9889.200195      1   0.002349
2025-12-24  9870.700195      1  -0.001872
2025-12-29  9866.500000      1  -0.

In [148]:
# Shift the data because we predict the next state at the close to avoid look ahead bias, we open/close a position with that day close price and accumulate profit/loss for the next day
vix_data_states['State'] = vix_data_states['State'].shift(1)

spy_data_states['State'] = spy_data_states['State'].shift(1)

n225_data_states['State'] = n225_data_states['State'].shift(1)


vix_data_states.dropna(inplace=True)
spy_data_states.dropna(inplace=True)
n225_data_states.dropna(inplace=True)

In [149]:
vix_data_states['Position'] = np.where(vix_data_states['State']==1,1,-1)
spy_data_states['Position'] = np.where(spy_data_states['State']==1,1,-1)
n225_data_states['Position'] = np.where(n225_data_states['State']==1,1,-1)

In [153]:
vix_data_states["Daily_Outcome_hmm"] = vix_data_states["State"] * vix_data_states['P&L_daily']
vix_data_states["Cumulative_Outcome_BaH"] = vix_data_states['P&L_daily'].cumsum()
vix_data_states['Cumulative_Outcome_hmm'] = vix_data_states['Daily_Outcome_hmm'].cumsum()


spy_data_states["Daily_Outcome_hmm"] = spy_data_states["State"] * spy_data_states['P&L_daily']
spy_data_states["Cumulative_Outcome_BaH"] = spy_data_states['P&L_daily'].cumsum()
spy_data_states['Cumulative_Outcome_hmm'] = spy_data_states['Daily_Outcome_hmm'].cumsum()


n225_data_states["Daily_Outcome_hmm"] = n225_data_states["State"] * n225_data_states['P&L_daily']
n225_data_states["Cumulative_Outcome_BaH"] = n225_data_states['P&L_daily'].cumsum()
n225_data_states['Cumulative_Outcome_hmm'] = n225_data_states['Daily_Outcome_hmm'].cumsum()

vix_data_states

,Close,State,P&L_daily,Position,Daily_Outcome_hmm,umulative_Outcome_BaH,Cumulative_Outcome_hmm,Cumulative_Outcome_BaH
Date,,,,,,,,
2016-01-22,22.340000,0.0,-0.177910,-1,-0.000000,-0.177910,-0.000000,-0.177910
2016-01-25,24.150000,0.0,0.077906,-1,0.000000,-0.100005,0.000000,-0.100005
2016-01-26,22.500000,0.0,-0.070769,-1,-0.000000,-0.170774,0.000000,-0.170774
2016-01-27,23.110001,0.0,0.026750,-1,0.000000,-0.144024,0.000000,-0.144024
2016-01-28,22.420000,1.0,-0.030312,1,-0.030312,-0.174336,-0.030312,-0.174336
...,...,...,...,...,...,...,...,...
2025-12-24,13.470000,1.0,-0.038592,1,-0.038592,-0.683824,4.801782,-0.683824
2025-12-26,13.600000,1.0,0.009605,1,0.009605,-0.674219,4.811387,-0.674219
2025-12-29,14.200000,1.0,0.043172,1,0.043172,-0.631047,4.854559,-0.631047


In [156]:
fig = go.Figure()

fig.add_trace(go.Line(x= vix_data_states.index, y=vix_data_states["Cumulative_Outcome_BaH"], 
                      name = 'Cumulative_Outcome_BaH', line_color = 'navy'))

fig.add_trace(go.Line(x= vix_data_states.index, y=vix_data_states['Cumulative_Outcome_hmm'], 
                      name = 'Cumulative_Outcome_hmm', line_color = 'olive'))

fig.update_layout(title="VIX",height=400, width=900, legend=dict(
    yanchor="top", y=0.99, xanchor="left",x=0.01), 
    margin=dict(l=20, r=20, t=20, b=20))

fig.show()

c:\Users\DELL 5520\anaconda3\Lib\site-packages\plotly\graph_objs\_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.




In [157]:
fig = go.Figure()

fig.add_trace(go.Line(x= spy_data_states.index, y=spy_data_states["Cumulative_Outcome_BaH"], 
                      name = 'Cumulative_Outcome_BaH', line_color = 'navy'))

fig.add_trace(go.Line(x= spy_data_states.index, y=spy_data_states['Cumulative_Outcome_hmm'], 
                      name = 'Cumulative_Outcome_hmm', line_color = 'olive'))

fig.update_layout(title="SPY",height=400, width=900, legend=dict(
    yanchor="top", y=0.99, xanchor="left",x=0.01), 
    margin=dict(l=20, r=20, t=20, b=20))

fig.show()

c:\Users\DELL 5520\anaconda3\Lib\site-packages\plotly\graph_objs\_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.




In [158]:
fig = go.Figure()

fig.add_trace(go.Line(x= n225_data_states.index, y=n225_data_states["Cumulative_Outcome_BaH"], 
                      name = 'Cumulative_Outcome_BaH', line_color = 'navy'))

fig.add_trace(go.Line(x= n225_data_states.index, y=n225_data_states['Cumulative_Outcome_hmm'], 
                      name = 'Cumulative_Outcome_hmm', line_color = 'olive'))

fig.update_layout(title="SPY",height=400, width=900, legend=dict(
    yanchor="top", y=0.99, xanchor="left",x=0.01), 
    margin=dict(l=20, r=20, t=20, b=20))

fig.show()

c:\Users\DELL 5520\anaconda3\Lib\site-packages\plotly\graph_objs\_deprecations.py:378: DeprecationWarning:

plotly.graph_objs.Line is deprecated.
Please replace it with one of the following more specific types
  - plotly.graph_objs.scatter.Line
  - plotly.graph_objs.layout.shape.Line
  - etc.


